# Imagenet 64 (64x64x3) -> Julia Memory Mapped IO

In [1]:
include("lib/imagenet.jl")
include("lib/utils.jl")

import .ImageNet
using .Utils

# Training Dataset

In [2]:
function load_train(files, output)
    function load_one(file::String)
        # N 64 64 3
        labels, images, μ = ImageNet.load(file, 64)
        # 64 64 3 N
        pimages = permutedims(images, (2, 3, 4, 1))
        return labels, pimages, μ
    end
        
    base = "build/$output"
    ispath(base) && rm(base, force=true, recursive=true)
    mkpath(base)
    
	open("$base/count.ser", "w") do io
		write(io, length(files))
	end

	ys = open("$base/ys.ser", "w")
	Xs = open("$base/Xs.ser", "w")
	μs = open("$base/μs.ser", "w")
	for (i, f) ∈ enumerate(files)
		ser = load_one(f)
		y, X, μ = ser
		serialize(Xs, X)
		serialize(μs, μ)
		serialize(ys, y)

		h, w, c, n = size(X)
		@info "input" file=f n=n
	end
	close(ys)
	close(Xs)
	close(μs)
end

@time begin
    load_train(vcat(
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part1", join=true),
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_train_part2", join=true)
    ), "train")
end

┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part1/train_data_batch_1"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part1/train_data_batch_2"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part1/train_data_batch_3"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part1/train_data_batch_4"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part1/train_data_batch_5"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part2/train_data_batch_10"
└   n = 128123
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part2/train_data_batch_6"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part2/train_data_batch_7"
└   n = 128116
┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_train_part2/train_data_batch_8"
└   n = 128116
┌ Info: i

195.875528 seconds (69.03 M allocations: 207.467 GiB, 4.80% gc time, 0.98% compilation time)


# Validation Dataset

In [5]:
function load_val(file, output)
    # N 64 64 3
    labels, images, _ = ImageNet.load(file, 64; has_mean=false)
    # 64 64 3 N
    pimages = permutedims(images, (2, 3, 4, 1))
        
    base = "build/$output"
    ispath(base) && rm(base, force=true, recursive=true)
    mkpath(base)
    
	ys = open("$base/ys.ser", "w")
	Xs = open("$base/Xs.ser", "w")
    serialize(Xs, pimages)
    serialize(ys, labels)
	close(ys)
	close(Xs)

    h, w, c, n = size(pimages)
    @info "input" file=file n=n
end

@time load_val(
        readdir("$(ImageNet.IMAGENET_64)/Imagenet64_val", join=true)[1],
        "val")

┌ Info: input
│   file = "/data/imagenet/Imagenet64/Imagenet64_val/val_data"
└   n = 50000


  7.319376 seconds (2.50 M allocations: 8.087 GiB, 5.66% gc time)
